In [1]:
import torch as t
from transformers import AutoModelForCausalLM, AutoModelForMaskedLM, AutoModelForImageTextToText, MistralForCausalLM, AutoModelForSequenceClassification
import argparse
import pandas as pd
from tqdm import tqdm
import os
from pprint import pprint
import configparser
from nnsight import LanguageModel

In [52]:
config = configparser.ConfigParser()
config.read('config.ini')

model_name = "Qwen3-8B"
n_heads = 32
head_dim = 128
weights_directory = config[model_name]['weights_directory']
model = LanguageModel(weights_directory, torch_dtype=t.bfloat16, device_map="mps")

In [53]:
statement = [
    """This is not a major issue because long running migrations are so infrequently run, but just wanted to call it out. OpenVoxDB will bind its port during startup, but before migrations are run, but it will reject all queries until 
    migrations are finished. So I think this will report the service as ready before it can respond to queries. This 
    may be a problem for catalog applications that apply a long running migration where subsequent resources Require 
    the db service and expect to be able to query it.""",
    """@user123 I was hoping we can use the unit file from the debian maintainers (#37 (comment)) as an intermediate solution because that's quicker to implement and test. As a long term solution I would also prefer the trapperkeeper update."""
]

In [70]:
layers = list(range(len(model.model.layers)))

acts = {}
tracer_kwargs = {'scan': False, 'validate': False}
with model.trace(statement, remote=False, **tracer_kwargs):
    for layer in layers:
        acts[layer] = {}
        attn_output = model.model.layers[layer].self_attn.o_proj.output.save()#.input[0][0].save()
        head_outputs = attn_output.view(attn_output.shape[0], attn_output.shape[1],n_heads, head_dim) #transform output to heads 
        for head in range(0,n_heads):
            acts[layer][head] = head_outputs[:, :, head, :].save()
for layer, heads in acts.items():
    for head, act in heads.items():
        acts[layer][head] = act.value
#print(layer)
#print(acts[-1])
print()
print("####################")


####################


In [71]:
acts[0][0].size()

torch.Size([2, 105, 128])

In [47]:
head_outputs = acts[0].view(-1, 32, 128)#[0,:,:]
head_outputs.size()

torch.Size([103, 32, 128])

In [48]:
head_0_output = head_outputs[:, 31, :]
head_0_output 

tensor([[-0.0221, -0.0124, -0.0199,  ...,  0.1016,  0.0247, -0.0022],
        [-0.0630, -0.0032, -0.0486,  ...,  0.0359,  0.0396,  0.0110],
        [-0.0170, -0.0184, -0.0432,  ...,  0.0737,  0.0664,  0.0271],
        ...,
        [ 0.0056,  0.0037, -0.0219,  ...,  0.0007,  0.0159, -0.0154],
        [-0.0126, -0.0287,  0.0076,  ...,  0.0088, -0.0109,  0.0437],
        [ 0.0071, -0.0471,  0.0081,  ...,  0.0123,  0.0060,  0.0262]],
       device='mps:0', dtype=torch.bfloat16, grad_fn=<SliceBackward0>)

In [49]:
head_0_output.size()

torch.Size([103, 128])